<a href="https://colab.research.google.com/github/MihaelaSandru/Seller-Data-Compliance/blob/main/Seller_Data_Compliance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# Import libraries and seller data
import pandas as pd
from datetime import datetime
import re
from google.colab import files
upload = files.upload()

Saving seller_data.csv to seller_data (3).csv


In [5]:
#import Amazon Supplier Agreement

from google.colab import files
upload = files.upload()

Saving Amazon_Services_Europe_Business_Solutions_Agreement_-_December_2022.pdf to Amazon_Services_Europe_Business_Solutions_Agreement_-_December_2022 (1).pdf


In [8]:
# Configuration of variables

INPUT_FILE = "seller_data.csv"
OUTPUT_FILE = "seller_validation_report.csv"

today = pd.Timestamp.today().normalize()

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

In [10]:
# Read CSV

df = pd.read_csv(INPUT_FILE)

date_columns_expected = [
    "id_expiry",
    "passport_expiry",
    "insurance_expiry",
    "credit_card_expiry",
    "last_account_update"
]

date_columns_actual = [col for col in date_columns_expected if col in df.columns]

for col in date_columns_actual:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [15]:
display(df.head())

,supplier_id,business_name,phone,insurance_expiry,email,supplier_zip_code_prefix,supplier_city,supplier_state,supplier_delivery_agreement
0,3442f8959a84dea7ee197c632cb2df15,ABC Ltd,(89) 91081-4185,2028-03-09,abc10@globo.com,13023,campinas,SP,3-4 days
1,d1b65fc7debc3361ea86b5f14c68d2e2,cba Ltd,(69) 91680-9389,2027-09-22,cba76@uol.com.br,13844,mogi guacu,SP,3-4 days
2,ce3ad9de960102d0677a81f5d0bb7b2d,Prime Services Inc,(11) 94823-1882,2026-01-16,contato@primeservices.com.br,20031,rio de janeiro,RJ,3-4 days
3,c0f3eea2e14555b6faeea3dd58c1b1c3,Smart Star Brasil,(34) 1882-8603,2026-11-26,financeiro@smartstar.com.br,4195,sao paulo,SP,3-4 days
4,51a04a8a6bdcb23deccc82b0b80742cf,Atlantic Point Goods,(38) 94895-3696,2027-01-08,atendimento@atlanticpointgoods.com.br,12914,braganca paulista,SP,3-4 days


In [25]:
import pandas as pd
from datetime import datetime
import re

# Configuration of variables (assuming these are defined in a previous cell)
# INPUT_FILE = "seller_data.csv"
# OUTPUT_FILE = "seller_validation_report.csv"
# today = pd.Timestamp.today().normalize()
# email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

# Validation Function

def validate(row):

    issues = []

    # These columns are assumed to always be present based on previous steps and df.columns
    if pd.isna(row["business_name"]) or str(row["business_name"]).strip() == "":
        issues.append("Missing Business Name")

    if pd.isna(row["email"]) or not re.match(email_regex, str(row["email"])):
        issues.append("Invalid Email")

    if pd.isna(row["phone"]) or len(str(row["phone"])) < 8:
        issues.append("Invalid Phone")



    # Expiration checks - only for 'insurance_expiry' as it's the only one present in df.columns
    if "insurance_expiry" in row.index:
        date = row["insurance_expiry"]
        if pd.isna(date):
            issues.append("Insurance Missing")
        elif date < today: # type: ignore
            issues.append("Insurance Expired")
        elif (date - today).days <= 30: # type: ignore
            issues.append("Insurance Expires Soon")

    # VAT validation integration
    if "VAT" in row.index:
        if "country" in row.index:
            vat_number = str(row["VAT"]).strip()
            country = str(row["country"]).strip().upper() # Convert country to uppercase for validator keys
            if not vat_number: # Check for empty string after strip
                issues.append("Missing VAT Number")
            else:
                # Assumes validate_company_number is defined in a previous cell
                validation_result = validate_company_number(country, vat_number)
                if validation_result != "Valid":
                    issues.append(f"VAT Validation: {validation_result}")
        else:
            # If VAT is present but country is not, add an issue indicating this dependency
            issues.append("Country column missing for VAT validation")
    # If "VAT" not in row.index, then we skip VAT validation as per previous refactoring.


    return "; ".join(issues) if issues else "OK"

In [21]:
# Run validation

df["Validation_Result"] = df.apply(validate, axis=1)

# Save report

df.to_csv(OUTPUT_FILE, index=False)

print(df[["supplier_id", "Validation_Result"]])

print(f"\nValidation report saved as {OUTPUT_FILE}")

                           supplier_id  Validation_Result
0     3442f8959a84dea7ee197c632cb2df15                 OK
1     d1b65fc7debc3361ea86b5f14c68d2e2                 OK
2     ce3ad9de960102d0677a81f5d0bb7b2d  Insurance Expired
3     c0f3eea2e14555b6faeea3dd58c1b1c3                 OK
4     51a04a8a6bdcb23deccc82b0b80742cf                 OK
...                                ...                ...
3090  98dddbc4601dd4443ca174359b237166                 OK
3091  f8201cab383e484733266d1906e2fdfa                 OK
3092  74871d19219c7d518d0090283e03c137                 OK
3093  e603cf3fec55f8697c9059638d6c8eb5                 OK
3094  9e25199f6ef7e7c347120ff175652c3b                 OK

[3095 rows x 2 columns]

Validation report saved as seller_validation_report.csv


In [26]:
validation_summary = df['Validation_Result'].value_counts()
display(validation_summary)

,count
Validation_Result,
OK,2370
Insurance Expired,625
Insurance Expires Soon,100


In [23]:
# VAT validator
import re

def validate_company_number(country, number):

    number = str(number).strip()

    validators = {

        "UK": r"^[A-Z0-9]{8}$",

        "DE": r"^[A-Z0-9]{6,20}$",

        "FR": r"^\d{9}$",           # SIREN

        "IT": r"^\d{11}$",          # Partita IVA

        "ES": r"^[A-Z0-9]{9}$",

        "NL": r"^\d{8}$",

        "BE": r"^\d{10}$",

        "PL": r"^\d{10}$"
    }

    if country not in validators:
        return "Country Not Supported"

    if re.match(validators[country], number):
        return "Valid"

    return "Invalid Format"